In [ ]:
%pip install dash
%pip install pandas
%pip install statsmodels

In [ ]:
from dash import Dash, html, dcc, Input, Output
import plotly.express as px
import pandas as pd
import json
import statistics

pd.options.plotting.backend = 'plotly' 

with open("./.data_Q2.txt", "r") as file:
    data_Q2 = json.loads(file.read())

with open("./.data_Q3.txt", "r") as file:
    data_Q3 = json.loads(file.read())


def filter_prec(data, threshold):
    teams_lt = []
    goals_lt = []
    teams_gt = []
    goals_gt =[]
    df1 = pd.DataFrame()
    df2 = pd.DataFrame()
    for team, matches in data.items():
        for match in list(zip(*matches)):
            if match[1] > threshold:
                teams_gt.append(team)
                goals_gt.append(match[0])
            else:
                teams_lt.append(team)
                goals_lt.append(match[0])
    df1["Teams"] = teams_lt
    df1["Goals"] = goals_lt
    df2["Teams"] = teams_gt
    df2["Goals"] = goals_gt
    return px.box(df1, x="Teams", y="Goals"), px.box(df2,x="Teams", y="Goals")
    

fig_Q2_0, fig_Q2_1 = filter_prec(data_Q2, 1.5)

fig_Q3_0 = px.scatter(data_Q3)
df = pd.DataFrame()
avgs = []
precs = []
for i in range(0, 350):
    matches = list(filter(lambda x: i/10 <= x[1] < i/10+0.1, list(zip(data_Q3["Goals"], data_Q3["Precipitation"]))))
    if len(matches) > 5:
        avgs.append(statistics.mean(x for (x, _) in matches))
        precs.append(i/10)

df["Average goals per match"] = avgs
df["Precipitation group"] = precs
fig_Q3_1 = px.scatter(df, x="Precipitation group",y="Average goals per match", trendline="ols")
# Initialize the app
app = Dash()

# App layout
app.layout = [
    html.P("Goals distribution for matches with less/more than the slider position's worth of precipitation"),
    dcc.Slider(
        id='rain_filter',
        min=0,
        max=35,
        step=0.1,
        value=1.5,
    ),
    html.Div(children=dcc.Graph(id= "Q2_0", figure=fig_Q2_0)),
    html.Div(children=dcc.Graph(id= "Q2_1", figure=fig_Q2_1)),
    html.Div(children=dcc.Graph(id= "Q3_0", figure=fig_Q3_0)),
    html.Div(children=dcc.Graph(id= "Q3_1", figure=fig_Q3_1)),
]

@app.callback(
    Output(component_id="Q2_0", component_property="figure"),
    Output(component_id="Q2_1", component_property="figure"),
    Input(component_id="rain_filter", component_property="value")
)
def update_plot(threshold):
    return filter_prec(data_Q2, threshold)


# Run the app
if __name__ == '__main__':
    app.run(debug=True, port = 8080)
